<a href="https://colab.research.google.com/github/krupa-16/AI/blob/main/unification_algorithm_140.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import re

# Variables are x, y, z, u, v, w (case-insensitive).
# Other names, such as A and B, are constants.
VARIABLES = {"x", "y", "z", "u", "v", "w"}


def is_variable(term):
    return isinstance(term, str) and term.lower() in VARIABLES


def parse_term(s):
    s = s.strip()

    # Normalize variable names so Y and y represent the same variable.
    match = re.fullmatch(r'([a-zA-Z_]\w*)\s*\((.*)\)', s)

    if not match:
        return s.lower() if is_variable(s) else s

    name, content = match.groups()
    args = []
    depth = 0
    start = 0

    for i, ch in enumerate(content):
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        elif ch == ',' and depth == 0:
            args.append(parse_term(content[start:i]))
            start = i + 1

    if content.strip():
        args.append(parse_term(content[start:]))

    return (name, args)


def apply_substitution(term, subst):
    if isinstance(term, str):
        while is_variable(term) and term in subst:
            term = subst[term]
        return term

    name, args = term
    return (name, [apply_substitution(a, subst) for a in args])


def occurs(var, term, subst):
    term = apply_substitution(term, subst)

    if term == var:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, arg, subst) for arg in term[1])

    return False


def unify(x, y, subst):
    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)

    if x == y:
        return True

    if is_variable(x):
        if occurs(x, y, subst):
            return False
        subst[x] = y
        return True

    if is_variable(y):
        if occurs(y, x, subst):
            return False
        subst[y] = x
        return True

    # Different constants cannot be unified.
    if isinstance(x, str) or isinstance(y, str):
        return False

    # Function names and arities must match.
    if x[0] != y[0] or len(x[1]) != len(y[1]):
        return False

    for a, b in zip(x[1], y[1]):
        if not unify(a, b, subst):
            return False

    return True


def format_term(term):
    if isinstance(term, str):
        return term

    name, args = term
    return name + "(" + ",".join(
        format_term(a) for a in args
    ) + ")"


def main():
    print("UNIFICATION ALGORITHM")

    p1 = input("Enter first predicate name: ").strip()
    a1 = input(
        "Enter arguments of first predicate (comma-separated): "
    )
    p2 = input("Enter second predicate name: ").strip()
    a2 = input(
        "Enter arguments of second predicate (comma-separated): "
    )

    if p1 != p2:
        print("FAILURE: Predicate symbols are different.")
        return

    expr1 = parse_term(p1 + "(" + a1 + ")")
    expr2 = parse_term(p2 + "(" + a2 + ")")

    if len(expr1[1]) != len(expr2[1]):
        print("FAILURE: Number of arguments is different.")
        return

    subst = {}

    if unify(expr1, expr2, subst):
        print("UNIFICATION SUCCESSFUL!")
        if subst:
            print("Substitution:")
            for var, value in subst.items():
                print(
                    var, "=",
                    format_term(apply_substitution(value, subst))
                )
        else:
            print("No substitution required.")
    else:
        print("FAILURE: Expressions cannot be unified.")


if __name__ == "__main__":
    main()


UNIFICATION ALGORITHM
Enter first predicate name: Q
Enter arguments of first predicate (comma-separated): Y,G(A,B)
Enter second predicate name: Q
Enter arguments of second predicate (comma-separated): G(x,x),y
FAILURE: Expressions cannot be unified.
